In [4]:
# Unique scores per fixture — load CSVs, merge, and list fixtures with exactly one correct predictor
import pandas as pd
from pathlib import Path
from IPython.display import display
import glob

ROOT = Path('..') / 'CSV'

def find_csv(name_patterns):
    matches = []
    for pat in name_patterns:
        matches.extend(glob.glob(str(ROOT / '**' / f'*{pat}*'), recursive=True))
    return Path(matches[0]) if matches else None

def load_csv(p):
    p = Path(p)
    if not p.exists():
        raise FileNotFoundError(f'File not found: {p}')
    df = pd.read_csv(p, dtype=str)
    df = df.apply(lambda col: col.str.strip() if col.dtype == 'object' else col)
    for c in df.columns:
        cl = c.lower()
        if any(k in cl for k in ('date','time','created_at','updated_at')):
            df[c] = pd.to_datetime(df[c], errors='coerce')
    return df

# locate files
profiles_path = find_csv(['profiles.csv','profiles'])
pred_path = find_csv(['predictions_rows.csv','predictions.csv','predictions_rows'])
fixtures_path = find_csv(['fixtures_rows.csv','fixtures.csv','fixtures_rows'])
gw_path = find_csv(['game_weeks_rows.csv','game_weeks.csv','game_weeks_rows'])

if not (profiles_path and pred_path and fixtures_path and gw_path):
    missing = [name for name,path in [('profiles',profiles_path),('predictions',pred_path),('fixtures',fixtures_path),('game_weeks',gw_path)] if path is None]
    raise FileNotFoundError('Missing CSV(s): ' + ', '.join(missing))

profiles = load_csv(profiles_path)
predictions = load_csv(pred_path)
fixtures = load_csv(fixtures_path)
gws = load_csv(gw_path)

# prefer common column names
# fixture actual scores expected in fixtures: home_score, away_score (sample confirmed)
for col in ('home_score','away_score'):
    if col not in fixtures.columns:
        raise KeyError(f"Expected '{col}' in fixtures CSV but not found. Found: {list(fixtures.columns)}")

# find predicted score column names in predictions (be permissive)
def find_pred_cols(df):
    cols = { 'home': None, 'away': None }
    for c in df.columns:
        lc = c.lower()
        if ('home' in lc) and ('pred' in lc or 'prediction' in lc or 'predicted' in lc):
            cols['home'] = c
        if ('away' in lc) and ('pred' in lc or 'prediction' in lc or 'predicted' in lc):
            cols['away'] = c
    # fallback: look for columns named 'home' and 'away' (less likely)
    if cols['home'] is None:
        for c in df.columns:
            if c.lower() == 'home_prediction' or c.lower() == 'home':
                cols['home'] = c
    if cols['away'] is None:
        for c in df.columns:
            if c.lower() == 'away_prediction' or c.lower() == 'away':
                cols['away'] = c
    return cols

pred_cols = find_pred_cols(predictions)
if not pred_cols['home'] or not pred_cols['away']:
    raise KeyError(f"Could not locate predicted home/away columns in predictions. Columns: {list(predictions.columns)}")

# Merge sequence: predictions -> fixtures -> game_weeks -> profiles
if 'fixture_id' not in predictions.columns:
    raise KeyError("predictions missing 'fixture_id' column")
if 'user_id' not in predictions.columns:
    raise KeyError("predictions missing 'user_id' column")

merged = predictions.merge(fixtures[['id','home_team','away_team','home_score','away_score','game_week_id']], left_on='fixture_id', right_on='id', how='left', suffixes=('','_fixture'))
merged = merged.merge(gws[['id','week_number','season_id']], left_on='game_week_id', right_on='id', how='left', suffixes=('','_gw'))
merged = merged.merge(profiles[['id','username']], left_on='user_id', right_on='id', how='left', suffixes=('','_profile'))

# remove test user 'Martinez' (case-insensitive)
if 'username' in merged.columns:
    merged = merged[merged['username'].fillna('').str.strip().str.lower() != 'martinez'].copy()

# normalize numeric scores
merged['__home_score'] = pd.to_numeric(merged['home_score'], errors='coerce')
merged['__away_score'] = pd.to_numeric(merged['away_score'], errors='coerce')
merged['__pred_home'] = pd.to_numeric(merged[pred_cols['home']], errors='coerce')
merged['__pred_away'] = pd.to_numeric(merged[pred_cols['away']], errors='coerce')

# exact match flag
merged['exact_match'] = (merged['__home_score'] == merged['__pred_home']) & (merged['__away_score'] == merged['__pred_away'])

# find fixtures where exactly one player predicted exact score
exact = merged[merged['exact_match']].copy()
counts = exact.groupby('fixture_id').agg(correct_count=('user_id','nunique')).reset_index()
unique_fixtures = counts[counts['correct_count'] == 1]['fixture_id'].tolist()

unique_rows = exact[exact['fixture_id'].isin(unique_fixtures)].copy()
# For each fixture there should be a single row; pick it
unique_rows = unique_rows.drop_duplicates(subset=['fixture_id'])

# prepare output columns
unique_rows['week_number'] = unique_rows['week_number']
unique_rows['fixture'] = unique_rows['home_team'].astype(str) + ' Vs. ' + unique_rows['away_team'].astype(str)
unique_rows['score'] = unique_rows['__home_score'].astype(int).astype(str) + '-' + unique_rows['__away_score'].astype(int).astype(str)
unique_rows['player'] = unique_rows['username']

out = unique_rows[['week_number','fixture','score','player']].copy()
# sort by week_number then fixture
try:
    out['week_sort'] = pd.to_numeric(out['week_number'], errors='coerce')
    out = out.sort_values(['week_sort','fixture'], na_position='last').drop(columns=['week_sort']).reset_index(drop=True)
except Exception:
    out = out.sort_values(['week_number','fixture']).reset_index(drop=True)

print(f'Found {len(out)} fixtures with a unique exact-score predictor')
display(out)


Found 42 fixtures with a unique exact-score predictor


,week_number,fixture,score,player
0,1,Man Utd Vs. Arsenal,0-1,Jim Shirley
1,1,Wolves Vs. Man City,0-4,Gerard
2,2,Brentford Vs. Aston Villa,1-0,Rod McGeady
3,2,Man City Vs. Tottenham,0-2,Martin H
4,3,Liverpool Vs. Arsenal,1-0,Sid Elliott
5,4,Newcastle Vs. Wolves,1-0,Jim Shirley
6,5,Bournemouth Vs. Newcastle,0-0,Bob sullivan
7,6,Crystal Palace Vs. Liverpool,2-1,Si B
8,6,Manchester City Vs. Burnley,5-1,Parish
9,6,Nottingham Forest Vs. Sunderland,0-1,Jim Shirley
